# Register Agent to Agentspace

@wangdave

### Prerequisites

- You have an Agentspace application deployed
- You have Agent Engines deployed
- Create a .env file (see below)
- You create OAuth 2.0 client with proper scopes and redirect URL:
https://vertexaisearch.cloud.google.com/oauth-redirect

Example below is based on [Google's OAuth 2.0 client](https://support.google.com/googleapi/answer/6158849):
  - From the GCP console, search for "OAuth consent screen"
  - Create an OAuth 2.0 client for Web application
  - Add the redirect URL specified above
  - Click **Create**
  - Download the `client_secret.json` and upload it here

For other 3rd party OAuth clients, follow their documentation to obtain authorization information.


### 0. Create .env if you don't have one
Fill in the following values in the `.env` file:

In [ ]:
from dotenv import load_dotenv
import IPython
import google_auth_oauthlib.flow
import json
import os
import vertexai

### Install required libraries

In [ ]:
!pip3 install -qq -U google-adk "google-cloud-aiplatform[agent_engines]" google-cloud-discoveryengine google-auth-oauthlib

### Restart Kernel

In [ ]:
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

## 1. Load Environment Variables and Initialize Vertex AI


In [ ]:
# take environment variables from .env.
load_dotenv()
PROJECT_ID = os.getenv("PROJECT_ID")
LOCATION = os.getenv("LOCATION", "us-central1")
GCS_BUCKET = os.getenv("GCS_BUCKET")
PROJECT_NUMBER = os.getenv("PROJECT_NUMBER")
#REASONING_ENGINE = os.getenv("REASONING_ENGINE")
AS_APP = os.getenv("AS_APP")
AUTH_ID = os.getenv("AUTH_ID")


In [ ]:
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=GCS_BUCKET,
)

In [ ]:
# your agent engine resource name
#REASONING_ENGINE

## 2. Generate OAuth 2.0 Authorization URL

In [ ]:
CLIENT_ID=os.getenv("CLIENT_ID")

CLIENT_SECRET=os.getenv("CLIENT_SECRET")
CLIENT_ID

In [ ]:
authorization_url=f"https://accounts.google.com/o/oauth2/v2/auth?client_id={CLIENT_ID}&redirect_uri=https%3A%2F%2Fvertexaisearch.cloud.google.com%2Fstatic%2Foauth%2Foauth.html&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fbigquery&include_granted_scopes=true&response_type=code&access_type=offline&prompt=consent"

In [ ]:
authorization_url

In [ ]:
os.environ["OAUTH_AUTH_URI"] = authorization_url


## 3. Get OAuth 2.0 Client ID and Secret
- Get `OAUTH_CLIENT_ID` and `OAUTH_CLIENT_SECRET` from the OAuth client setup in the client_secret.json file
- Ensure that `OAUTH_TOKEN_URI` is what your service requires. Example used here is for Google APIs.

In [ ]:
client_id = os.getenv("CLIENT_ID")
client_secret = os.getenv("CLIENT_SECRET")

In [ ]:
!echo $CLIENT_ID

In [ ]:
os.environ["OAUTH_CLIENT_ID"] = client_id
os.environ["OAUTH_CLIENT_SECRET"] = client_secret
os.environ["OAUTH_TOKEN_URI"] = "https://oauth2.googleapis.com/token"

In [ ]:
%%bash

curl -X POST \
  -H "Authorization: Bearer $(gcloud auth print-access-token)" \
  -H "Content-Type: application/json" \
  -H "X-Goog-User-Project: ${PROJECT_NUMBER}" \
https://discoveryengine.googleapis.com/v1alpha/projects/${PROJECT_NUMBER}/locations/global/authorizations?authorizationId=${AUTH_ID} \
  -d '{
  "name": "projects/${PROJECT_NUMBER}/locations/global/authorizations/${AUTH_ID}",
  "serverSideOauth2": {
      "clientId": "'"${CLIENT_ID}"'",
      "clientSecret": "'"${CLIENT_SECRET}"'",
      "authorizationUri": "'"${OAUTH_AUTH_URI}"'",
      "tokenUri": "'"${OAUTH_TOKEN_URI}"'"
    }
  }'

## 4. Link Agent to Agentspace

In [ ]:
AGENT_URL=os.getenv("AGENT_URL")

In [ ]:
import requests
import subprocess

# Get the auth token
auth_token = subprocess.run(
    ["gcloud", "auth", "print-access-token"],
    capture_output=True,
    text=True
).stdout.strip()

# Construct the agent card as a proper JSON object first, then convert to string
agent_card = {
    "url": AGENT_URL,
    "name": "Test_Agent",
    "description": "You're an export of weather and cocktail, answer questions regarding weather and cocktail",
    "capabilities": {},
    "defaultInputModes": ["text/plain"],
    "defaultOutputModes": ["text/plain"],
    "skills": [],
    "version": "1.0.0",
    "protocolVersion": "0.1"
}

# Construct the full payload
payload = {
    "name": "A2A",
    "displayName": os.getenv("DISPLAY_NAME", "A2A Agent"),
    "description": os.getenv("DESCRIPTION", "A2A Agent Description"),
    "a2aAgentDefinition": {
        "jsonAgentCard": json.dumps(agent_card)
    },
    "authorization_config": {
        "agent_authorization": f"projects/{PROJECT_NUMBER}/locations/global/authorizations/{AUTH_ID}"
    }
}

# Make the request
url = f"https://discoveryengine.googleapis.com/v1alpha/projects/{PROJECT_NUMBER}/locations/global/collections/default_collection/engines/{AS_APP}/assistants/default_assistant/agents"

headers = {
    "Authorization": f"Bearer {auth_token}",
    "Content-Type": "application/json"
}

response = requests.post(url, json=payload, headers=headers)

print(f"Status Code: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

## 5. Try your agent in Agentspace UI
1. Ensure that `Vertex AI API` and `Discovery Engine API` are enabled in your project and that Discovery Engine service account has `Vertex AI User` permission. If you cannot find it in IAM, check the `Include Google-provided role grants`.
2. Open up the Agentspace app in Google Cloud console
3. Select your Agentspace app
3. Click `Copy URL`
4. Open the URL in a new tab
5. From left menu, select your agent from `Agents` deployed

## 6. List agents (Optional)

In [ ]:
%%bash
# export PROJECT_NUMBER=PROJECT_NUMBER
# export AS_APP=AGENTSAPCE_APP_ID
curl -X GET -H "Authorization: Bearer $(gcloud auth print-access-token)" \
-H "Content-Type: application/json" \
-H "X-Goog-User-Project: ${PROJECT_NUMBER}" \
"https://discoveryengine.googleapis.com/v1alpha/projects/${PROJECT_NUMBER}/locations/global/collections/default_collection/engines/${AS_APP}/assistants/default_assistant/agents"

## 7. Delete an agent (when you don't need it anymore )

In [ ]:
os.environ["AGENT_RESOURCE_NAME"] = "your agent resource name from the response"

In [ ]:
%%bash
# export PROJECT_NUMBER=PROJECT_NUMBER
# export AGENT_RESOURCE_NAME=AGENT_RESOURCE_NAME

curl -X DELETE \
  -H "Authorization: Bearer $(gcloud auth print-access-token)" \
  -H "Content-Type: application/json" \
  -H "X-Goog-User-Project: ${PROJECT_NUMBER}" \
https://discoveryengine.googleapis.com/v1alpha/${AGENT_RESOURCE_NAME}